In [22]:
import torch
import os
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download, login
import json
from IPython.display import Markdown, display

# 1. Extracting Data and formating

In [2]:
# read the data from json file
file_path = 'raw_ownship_intruders_data_Oct18.jsonl'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

In [4]:
data[:1]  # Display the first two entries to verify the data structure

[{'ownship_info': {'agent_id': 'AA0001',
   'ac_type': 'Google X-Wing',
   'ac_lat': 33.099264,
   'ac_lon': -96.895961,
   'next_wpt_id': 'WP3',
   'next_wpt_type': 'Regular',
   'ac_dist_to_nxt_wpt(m)': 3522.06,
   'speed(m/s)': 15.95,
   'ac_min_spd(m/s)': 0.0,
   'ac_max_spd(m/s)': 30.87,
   'speed_change_per_second(m/s2)': 1.03,
   'heading(deg)': 21.01,
   'heading_scale': '{North=0, East=90, South=180, West=270}',
   'altitude(m)': 1389.36,
   'ac_route_id': 'R_1',
   'ac_last_action': 'accelerate',
   'num_intruders_ahead': 0,
   'intruder1_on_same_route': None,
   'intruder2_on_same_route': None,
   'distance_to_intruder1(m)': None,
   'distance_to_intruder2(m)': None},
  'first_closest_front_intruder_info': [],
  'second_closest_front_intruder_info': [],
  'ownship_action': 'Accelerate'}]

In [13]:
# filter out entries with num_intruders_ahead=2
filtered_data = [entry for entry in data if entry['ownship_info'].get('num_intruders_ahead') == 2]
print(f"Total entries: {len(filtered_data)}")
firstentry = filtered_data[:1]  # Display the first entry of the filtered data to verify the filtering process
print(json.dumps(firstentry, indent=2))

Total entries: 20690
[
  {
    "ownship_info": {
      "agent_id": "AF0002",
      "ac_type": "Google X-Wing",
      "ac_lat": 33.179946,
      "ac_lon": -96.771063,
      "next_wpt_id": "WP7",
      "next_wpt_type": "Regular",
      "ac_dist_to_nxt_wpt(m)": 5439.98,
      "speed(m/s)": 12.86,
      "ac_min_spd(m/s)": 0.0,
      "ac_max_spd(m/s)": 30.87,
      "speed_change_per_second(m/s2)": 1.03,
      "heading(deg)": 269.54,
      "heading_scale": "{North=0, East=90, South=180, West=270}",
      "altitude(m)": 1389.36,
      "ac_route_id": "R_6",
      "ac_last_action": 1,
      "num_intruders_ahead": 2,
      "intruder1_on_same_route": true,
      "intruder2_on_same_route": false,
      "distance_to_intruder1(m)": 1644.03,
      "distance_to_intruder2(m)": 2034.01
    },
    "first_closest_front_intruder_info": {
      "agent_id": "AF0001",
      "ac_type": "Google X-Wing",
      "ac_lat": 33.179827,
      "ac_lon": -96.788727,
      "next_wpt_id": "WP7",
      "next_wpt_type": "Re

In [15]:
# data reformating for LLM input: 
# ownship_info: [v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2]
"""
v: ownship speed
hdg: ownship heading
delta_d1: signed distance to intruder 1
delta_d2: signed distance to intruder 2
delta_v1: ownship's velocity relative to intruder 1
delta_v2: ownship's velocity relative to intruder 2
delta_hdg1: change in heading to intruder 1
delta_hdg2: change in heading to intruder 2
"""
def reformat_entry(entry):
    ownship_info = entry['ownship_info']
    v = ownship_info.get('speed(m/s)', 0)
    own_hdg = ownship_info.get('heading(deg)', 0)
    own_d_to_intersection = ownship_info.get('ac_dist_to_nxt_wpt(m)', 0)
    rule_based_action = ownship_info.get('ownship_action', 'N/A')

    intruder1_info = entry['first_closest_front_intruder_info']
    intr1_v = intruder1_info.get('speed(m/s)', 0)
    intr1_hdg = intruder1_info.get('heading(deg)', 0)
    intr1_d_to_intersection = intruder1_info.get('ac_dist_to_nxt_wpt(m)', 0)

    intruder2_info = entry['second_closest_front_intruder_info']
    intr2_v = intruder2_info.get('speed(m/s)', 0)
    intr2_hdg = intruder2_info.get('heading(deg)', 0)
    intr2_d_to_intersection = intruder2_info.get('ac_dist_to_nxt_wpt(m)', 0)

    delta_d1 = own_d_to_intersection - intr1_d_to_intersection
    delta_d2 = own_d_to_intersection - intr2_d_to_intersection
    delta_v1 = v - intr1_v
    delta_v2 = v - intr2_v
    delta_hdg1 = own_hdg - intr1_hdg
    delta_hdg2 = own_hdg - intr2_hdg

    reformatted_entry = {
        'v': v,
        'hdg': own_hdg,
        'delta_d1': delta_d1,
        'delta_d2': delta_d2,
        'delta_v1': delta_v1,
        'delta_v2': delta_v2,
        'delta_hdg1': delta_hdg1,
        'delta_hdg2': delta_hdg2,
        'rule_based_action': rule_based_action
    }
    
    return reformatted_entry

In [19]:
system_prompt = """
You are an airspace tactical deconfliction assistant. 
Giving the ownship state: [v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2]
v: ownship speed(m/s), hdg: ownship heading(deg), 
delta_d1: signed distance difference from ownship to the first intruder (m) relative to next intersection, 
delta_d2: signed distance difference from ownship to the second intruder (m) relative to next intersection, 
delta_v1: speed difference from ownship to the first intruder (m/s), 
delta_v2: speed difference from ownship to the second intruder (m/s), 
delta_hdg1: heading difference from ownship to the first intruder (deg), 
delta_hdg2: heading difference from ownship to the second intruder (deg).
Decide the ownship action: Decelerate (-1.7m/s)/ Hold (+0 m/s)/ Accelerate (+1.7m/s).
Your response should be one of the three actions: "Decelerate", "Hold", or "Accelerate".
 """

In [20]:
user_prompt_template = """Given the ownship state: 
[v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2] = [{v}, {hdg}, {delta_d1}, {delta_d2}, {delta_v1}, {delta_v2}, {delta_hdg1}, {delta_hdg2}]
What is the recommended ownship action? Skip the explanation.
"""

In [17]:
input_entry = reformat_entry(firstentry[0])
input_entry

{'v': 12.86,
 'hdg': 269.54,
 'delta_d1': 1644.0299999999997,
 'delta_d2': 421.2999999999993,
 'delta_v1': -12.350000000000001,
 'delta_v2': 0.0,
 'delta_hdg1': 0.010000000000047748,
 'delta_hdg2': 21.960000000000008,
 'rule_based_action': 'N/A'}

In [21]:

message_template = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_template.format(
        v=input_entry['v'],
        hdg=input_entry['hdg'],
        delta_d1=input_entry['delta_d1'],
        delta_d2=input_entry['delta_d2'],
        delta_v1=input_entry['delta_v1'],
        delta_v2=input_entry['delta_v2'],
        delta_hdg1=input_entry['delta_hdg1'],
        delta_hdg2=input_entry['delta_hdg2']
    )}
]

print("Message Template:")
for message in message_template:
    print(f"{message['role'].capitalize()}: {message['content']}\n")


Message Template:
System: 
You are an airspace tactical deconfliction assistant. 
Giving the ownship state: [v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2]
v: ownship speed(m/s), hdg: ownship heading(deg), 
delta_d1: signed distance difference from ownship to the first intruder (m) relative to next intersection, 
delta_d2: signed distance difference from ownship to the second intruder (m) relative to next intersection, 
delta_v1: speed difference from ownship to the first intruder (m/s), 
delta_v2: speed difference from ownship to the second intruder (m/s), 
delta_hdg1: heading difference from ownship to the first intruder (deg), 
delta_hdg2: heading difference from ownship to the second intruder (deg).
Decide the ownship action: Decelerate (-1.7m/s)/ Hold (+0 m/s)/ Accelerate (+1.7m/s).
Your response should be one of the three actions: "Decelerate", "Hold", or "Accelerate".
 

User: Given the ownship state: 
[v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, del

# 2. Model loading and preparation for inference

In [ ]:
def gpu_memory():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        stdout=subprocess.PIPE,
        encoding="utf-8"
    )
    lines = result.stdout.strip().split("\n")
    for idx, line in enumerate(lines):
        used, total = line.split(", ")
        print(f"GPU {idx}: {used} MB / {total} MB used")

print("🔎 Before model load:")

gpu_memory()

MODEL_ID = "Qwen/Qwen2.5-Math-7B"
SHARED_MODEL_DIR = "/data/shared_LLM_models/Qwen2.5-Math-7B"
os.makedirs(SHARED_MODEL_DIR, exist_ok=True)
if not os.listdir(SHARED_MODEL_DIR):
    print(f"📦 Downloading {MODEL_ID} to {SHARED_MODEL_DIR} ...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=SHARED_MODEL_DIR,
        resume_download=True,
        local_dir_use_symlinks=False  # ensures all files are copied physically
    )
else:
    print(f"✅ Model already exists at {SHARED_MODEL_DIR}, skipping download.")


print(f"🚀 Loading model from: {SHARED_MODEL_DIR}")

tokenizer = AutoTokenizer.from_pretrained(SHARED_MODEL_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    SHARED_MODEL_DIR,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
).eval()

print("✅ Model ready for inference!")

# model device
print("🔎 After model load:")
print("Model device:", model.device)

## a. one entry testing

In [ ]:
# tokenize the input messages for the model
inputs = tokenizer.apply_chat_template(
    message_template,
    tokenize=True,
    add_generation_prompt=True,   # adds the assistant turn header
    return_tensors="pt"
).to(model.device)

gen_kwargs = {
    "max_new_tokens": 1000,
    "temperature": 0.2,
    "top_p": 0.95,
    "do_sample": False,  # start with greedy to verify it responds
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.eos_token_id,
    "use_cache": True,
}

with torch.no_grad():
    output_ids = model.generate(inputs, **gen_kwargs)
    
# Slice off the prompt tokens before decoding
gen_ids = output_ids[0, inputs.shape[-1]:]
text_output = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

display(Markdown(text_output))


## b. batch of data

In [25]:
# top 10 filtered data
filtered_data_top10 = filtered_data[:10]
filered_data_top10_reformatted = [reformat_entry(entry) for entry in filtered_data_top10]

In [26]:
filered_data_top10_reformatted

[{'v': 12.86,
  'hdg': 269.54,
  'delta_d1': 1644.0299999999997,
  'delta_d2': 421.2999999999993,
  'delta_v1': -12.350000000000001,
  'delta_v2': 0.0,
  'delta_hdg1': 0.010000000000047748,
  'delta_hdg2': 21.960000000000008,
  'rule_based_action': 'N/A'},
 {'v': 12.86,
  'hdg': 247.58,
  'delta_d1': 1644.0200000000004,
  'delta_d2': 1222.7300000000005,
  'delta_v1': -12.350000000000001,
  'delta_v2': -12.350000000000001,
  'delta_hdg1': 0.010000000000019327,
  'delta_hdg2': -21.94999999999996,
  'rule_based_action': 'N/A'},
 {'v': 15.95,
  'hdg': 269.54,
  'delta_d1': 1678.9900000000002,
  'delta_d2': 421.28999999999996,
  'delta_v1': -9.260000000000002,
  'delta_v2': 0.0,
  'delta_hdg1': 0.010000000000047748,
  'delta_hdg2': 21.960000000000008,
  'rule_based_action': 'N/A'},
 {'v': 15.95,
  'hdg': 247.58,
  'delta_d1': 1678.9900000000002,
  'delta_d2': 1257.7000000000003,
  'delta_v1': -9.260000000000002,
  'delta_v2': -9.260000000000002,
  'delta_hdg1': 0.010000000000019327,
  'delt

In [28]:


message_template_batch = [
    {"role": "system", "content": system_prompt},
]
message_template_batch.extend([{"role": "user", "content": user_prompt_template.format(
        v=input_entry['v'],
        hdg=input_entry['hdg'],
        delta_d1=input_entry['delta_d1'],
        delta_d2=input_entry['delta_d2'],
        delta_v1=input_entry['delta_v1'],
        delta_v2=input_entry['delta_v2'],
        delta_hdg1=input_entry['delta_hdg1'],
        delta_hdg2=input_entry['delta_hdg2']
    )} for input_entry in filered_data_top10_reformatted])

print("Message Template:")
for message in message_template_batch:
    print(f"{message['role'].capitalize()}: {message['content']}\n")

Message Template:
System: 
You are an airspace tactical deconfliction assistant. 
Giving the ownship state: [v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2]
v: ownship speed(m/s), hdg: ownship heading(deg), 
delta_d1: signed distance difference from ownship to the first intruder (m) relative to next intersection, 
delta_d2: signed distance difference from ownship to the second intruder (m) relative to next intersection, 
delta_v1: speed difference from ownship to the first intruder (m/s), 
delta_v2: speed difference from ownship to the second intruder (m/s), 
delta_hdg1: heading difference from ownship to the first intruder (deg), 
delta_hdg2: heading difference from ownship to the second intruder (deg).
Decide the ownship action: Decelerate (-1.7m/s)/ Hold (+0 m/s)/ Accelerate (+1.7m/s).
Your response should be one of the three actions: "Decelerate", "Hold", or "Accelerate".
 

User: Given the ownship state: 
[v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, del

In [ ]:
rendered_prompts = [
    tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
    for m in messages_batch
]
# --- Tokenize as one padded batch ---
enc = tokenizer(
    rendered_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(model.device)

In [ ]:
gen_kwargs = dict(
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    use_cache=True,
)
with torch.inference_mode():
    out = model.generate(**enc, **gen_kwargs)

with torch.inference_mode():
    out = model.generate(**enc, **gen_kwargs)
# --- Decode per row and extract final action ---
input_lens = (enc["input_ids"] != tokenizer.pad_token_id).sum(dim=1)
actions = []
for i in range(len(rendered_prompts)):
    gen_ids = out[i, input_lens[i]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    display(Markdown(text))